In [ ]:
import pandas as pd
import numpy as np

# Cargar archivos
admissions = pd.read_csv('D:/Universidad/5to Semestre/Inteligencia Artificial/Primer Parcial/Datasets/D2/mimic-iii-clinical-database-demo-1.4/ADMISSIONS.csv')
patients = pd.read_csv('D:/Universidad/5to Semestre/Inteligencia Artificial/Primer Parcial/Datasets/D2/mimic-iii-clinical-database-demo-1.4/PATIENTS.csv')
labs = pd.read_csv('D:/Universidad/5to Semestre/Inteligencia Artificial/Primer Parcial/Datasets/D2/mimic-iii-clinical-database-demo-1.4/LABEVENTS.csv')
diagnoses = pd.read_csv('D:/Universidad/5to Semestre/Inteligencia Artificial/Primer Parcial/Datasets/D2/mimic-iii-clinical-database-demo-1.4/DIAGNOSES_ICD.csv')

print(f"Filas originales en Labs: {len(labs)}")
 
# Sacamos promedio, valor mínimo y máximo de los resultados numéricos por cada ingreso.
lab_resumen = labs.groupby('hadm_id').agg({
    'valuenum': ['mean', 'min', 'max', 'count'],
    'flag': lambda x: (x == 'abnormal').sum()  # Contamos cuántos resultados fueron anormales
}).reset_index()

# Renombrar columnas para que sean claras
lab_resumen.columns = ['hadm_id', 'lab_mean', 'lab_min', 'lab_max', 'lab_count', 'lab_abnormal_count']

# Contamos cuántas enfermedades distintas le diagnosticaron en ese ingreso
diag_resumen = diagnoses.groupby('hadm_id').size().reset_index(name='total_diseases')

# Empezamos con admissions porque es nuestra unidad de análisis
df = pd.merge(admissions, patients[['subject_id', 'gender', 'dob']], on='subject_id', how='left')

# Unimos los resúmenes
df = pd.merge(df, lab_resumen, on='hadm_id', how='left')
df = pd.merge(df, diag_resumen, on='hadm_id', how='left')


# Si un paciente no tuvo laboratorios, sus valores serán NaN. 
# Para el modelo, los conteos deben ser 0.
df['lab_count'] = df['lab_count'].fillna(0)
df['lab_abnormal_count'] = df['lab_abnormal_count'].fillna(0)
df['total_diseases'] = df['total_diseases'].fillna(0)

# Para los valores promedio/min/max, usamos la mediana del resto de pacientes
cols_clinicas = ['lab_mean', 'lab_min', 'lab_max']
for col in cols_clinicas:
    df[col] = df[col].fillna(df[col].median())


df.to_csv('D:/Universidad/5to Semestre/Inteligencia Artificial/Primer Parcial/Datasets/D2/mimic_pro_full.csv', index=False)

# Versión para modelo (solo números)

cols_model = [c for c in df.columns if c not in ['row_id', 'subject_id', 'hadm_id', 'admittime', 
                                                'dischtime', 'deathtime', 'dob', 'diagnosis', 
                                                'hospital_expire_flag']]

df_final = pd.get_dummies(df[cols_model], drop_first=True)
df_final.to_csv('D:/Universidad/5to Semestre/Inteligencia Artificial/Primer Parcial/Datasets/D2/mimic_pro_optimized.csv', index=False)

print(f"Proceso terminado. Filas finales: {len(df)}")
print("Ahora cada fila contiene el resumen de miles de eventos de laboratorio.")